In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1: camera intrinsics calibration

In [9]:
# ChArUco board:
COLS, ROWS = 6, 8                       # squares across, down
square_len = 0.030                      # 30 mm squares, in metres
marker_len = 0.0225                     # 22.5 mm markers (must be < square)
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_100)
board = cv2.aruco.CharucoBoard((COLS, ROWS), square_len, marker_len, aruco_dict)
MARGIN = 60
cv2.imwrite("charuco_6x8.png", board.generateImage((960 + 2 * MARGIN, 1280 + 2 * MARGIN), marginSize=MARGIN))

# Print it, then MEASURE a printed square with calipers (span all 6 across,
# divide by 6) and set square_len from that - printers rescale, and every
# metric distance you estimate later is proportional to this number.

True

In [ ]:
def calibrate_camera_intrinsic(image_filenames):
    detector = cv2.aruco.CharucoDetector(board)   # sub-pixel refines internally
    all_obj, all_img = [], []
    image_resolution = None

    for img_file in image_filenames:
        img_bgr = cv2.imread(img_file)
        if img_bgr is None:
            print("=> unreadable: {0}".format(img_file))
            continue
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        wh = img_gray.shape[:2][::-1]   # converts numpy (h, w) to (w, h)
        if image_resolution is None:
            image_resolution = wh
        elif wh != image_resolution:
            raise ValueError("{0}: size {1} != {2}".format(img_file, wh, image_resolution))

        charucoCorners, charucoIDs, _, _ = detector.detectBoard(img_gray)
        n = 0 if charucoCorners is None else len(charucoCorners)
        print("=> Processing image {0}: {1} corners".format(img_file, n))
        objPoints, imgPoints = board.matchImagePoints(charucoCorners, charucoIDs)
        all_obj.append(objPoints)
        all_img.append(imgPoints)

    if len(all_obj) < 4:
        raise RuntimeError("only {0} usable views; aim for 20+".format(len(all_obj)))

    # Default 5-coefficient model (k1,k2,p1,p2,k3). CALIB_RATIONAL_MODEL adds 3
    # more radial terms but overfits unless many views reach the image border.
    (ret, camera_matrix, distortion_coefficients,
     rotation_vectors, translation_vectors) = cv2.calibrateCamera(
        all_obj, all_img, image_resolution, None, None)

    print('views used:', len(all_obj))
    print('ret (RMS reprojection, px):', ret)   # want < 0.5
    print('camera_matrix:', camera_matrix)
    print('distortion_coefficients:', distortion_coefficients.ravel())
    return True, camera_matrix, distortion_coefficients

def detect_calib_board(image_rgb, camera_matrix, dist_coeffs):
    output_img_rgb = image_rgb.copy()
    image_gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)

    detector = cv2.aruco.CharucoDetector(board)
    charucoCorners, chIds, mCorners, mIds = detector.detectBoard(image_gray)
    if charucoCorners is None or len(charucoCorners) < 4:
        print('Error: Could not identify the calibration board in the image!')
        return False, None, None, output_img_rgb

    if mIds is not None:
        cv2.aruco.drawDetectedMarkers(output_img_rgb, mCorners, mIds)
    cv2.aruco.drawDetectedCornersCharuco(output_img_rgb, charucoCorners, chIds)

    objPoints, imgPoints = board.matchImagePoints(charucoCorners, chIds)
    ret_val, board_rvec_rdg, board_tvec = cv2.solvePnP(
        objPoints, imgPoints, camera_matrix, dist_coeffs)
    if not ret_val:
        print('Error: Could not identity the pose of the calibration board in the image!')
        return False, None, None, output_img_rgb

    print('Board rvec: ')
    print(board_rvec_rdg)
    print('Board tvec: ')
    print(board_tvec)

    length_of_axis = 0.05          # metres; ~1.7 squares, visible at 30 mm pitch
    cv2.drawFrameAxes(output_img_rgb, camera_matrix, dist_coeffs,
                      board_rvec_rdg, board_tvec, length_of_axis)
    plt.imshow(output_img_rgb)
    plt.show()
    return True, board_rvec_rdg, board_tvec, output_img_rgb

In [16]:
# Live segmentation mask
cap = cv2.VideoCapture(1)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (12, 12))
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 90, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    cv2.imshow("mask", mask)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
# Live SVD drone pose estimator
cap = cv2.VideoCapture(0)
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (12, 12))
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 90, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt = max(contours, key=cv2.contourArea) # pick largest contour by area

        M = cv2.moments(cnt) # contour centroid
        if M["m00"] != 0:
            centroid_x = int(M["m10"] / M["m00"])
            centroid_y = int(M["m01"] / M["m00"])
        else:
            centroid_x = centroid_y = None

        print("centroid:", (centroid_x, centroid_y))

        if cv2.contourArea(cnt) > 200:
            fill = np.zeros_like(gray)
            cv2.drawContours(fill, [cnt], -1, 255, cv2.FILLED)
            ys, xs = np.nonzero(fill)
            pts = np.column_stack((xs, ys)).astype(float)
            center = pts.mean(axis=0)
            _, S, Vt = np.linalg.svd(pts - center, full_matrices=False)
            major = Vt[0]                                  # in-plane axis (image coords)
            alpha = np.arccos(np.clip(S[1] / S[0], 0, 1))  # tilt: minor/major = cos(alpha)
            normal = np.array([-np.sin(alpha) * major[1],  # disk-plane normal (3D)
                                np.sin(alpha) * major[0],
                                np.cos(alpha)])
            proj = (pts - center) @ major
            L = (proj.max() - proj.min()) / 2              # draw length from blob extent
            c = center.astype(int)
            cv2.line(frame, tuple((center - major * L).astype(int)),
                            tuple((center + major * L).astype(int)), (255, 0, 0), 2)
            cv2.line(frame, tuple(c),
                            tuple((center + normal[:2] * L).astype(int)), (0, 255, 0), 2)
            cv2.circle(frame, tuple(c), 4, (0, 0, 255), -1)
            image = cv2.circle(frame, (centroid_x, centroid_y), 4, (100, 0, 255), -1)


    cv2.imshow("pose", frame)
    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break
cap.release()
cv2.destroyAllWindows()

2026-07-25 22:16:41.361 Python[98236:6341742] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/63/7cv2q0897zv367k7c1bdmp500000gn/T/org.python.python.savedState


centroid: (395, 523)
centroid: (850, 436)
centroid: (904, 470)
centroid: (908, 479)
centroid: (903, 481)
centroid: (903, 481)
centroid: (903, 481)
centroid: (903, 481)
centroid: (902, 480)
centroid: (904, 481)
centroid: (901, 480)
centroid: (904, 480)
centroid: (903, 480)
centroid: (903, 481)
centroid: (905, 481)
centroid: (905, 482)
centroid: (906, 482)
centroid: (906, 483)
centroid: (905, 483)
centroid: (907, 483)
centroid: (943, 507)
centroid: (910, 488)
centroid: (943, 507)
centroid: (942, 507)
centroid: (943, 508)
centroid: (943, 508)
centroid: (942, 505)
centroid: (943, 507)
centroid: (942, 508)
centroid: (942, 509)
centroid: (942, 510)
centroid: (942, 511)
centroid: (942, 511)
centroid: (942, 512)
centroid: (942, 512)
centroid: (943, 512)
centroid: (942, 513)
centroid: (942, 513)
centroid: (942, 514)
centroid: (942, 514)
centroid: (942, 514)
centroid: (942, 512)
centroid: (942, 511)
centroid: (942, 510)
centroid: (942, 509)
centroid: (942, 509)
centroid: (942, 508)
centroid: (94

KeyboardInterrupt: 

: 